# Week 10 — Day 3 : Agentic AI with RAG (Retrieval-Augmented Generation)

**Bootcamp GenAI & Machine Learning 2026 — Developers Institute**

Build a tiny agent that:
1. Maintains an **in-memory knowledge base** (FAISS vector store)
2. Can query **Wikipedia** as a free external tool
3. Uses a **rule-based planner** to route questions
4. **Cites sources** and handles missing evidence gracefully

```
Question
  └──► Planner ──► KB retriever  [kb:source]
              └──► Wikipedia     [wiki:title]
                        │
                        ▼
                  Answer + citations
```

---
## Setup

In [ ]:
!pip install -q langchain langchain-community faiss-cpu wikipedia transformers accelerate sentencepiece

In [ ]:
import warnings
warnings.filterwarnings("ignore")

# Optional: use a tiny HF pipeline instead of FakeListLLM
USE_REAL_LLM = False  # Set True to load sshleifer/tiny-gpt2 (requires GPU/CPU time)

print("Setup complete ✅")

---
## Exercise 1 — Build the KB Retriever

Create 5–8 `Document` objects covering AI/ML topics, then build a FAISS retriever using `FakeEmbeddings`.

In [ ]:
from langchain_core.documents import Document
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings.fake import FakeEmbeddings

# --- 8 knowledge base documents ---
KB_DOCS = [
    Document(
        page_content=(
            "RAG (Retrieval-Augmented Generation) is a technique that combines a retrieval system "
            "with a generative language model. Instead of relying solely on the model's parametric "
            "memory, RAG fetches relevant documents from an external knowledge base and uses them as "
            "context to generate more accurate, grounded answers."
        ),
        metadata={"source": "rag_overview", "topic": "RAG"}
    ),
    Document(
        page_content=(
            "LangChain is an open-source framework for building applications powered by language models. "
            "It provides abstractions for chains, agents, retrievers, memory, and tool use. "
            "LangChain supports LCEL (LangChain Expression Language) for composing pipelines with the | operator."
        ),
        metadata={"source": "langchain_intro", "topic": "LangChain"}
    ),
    Document(
        page_content=(
            "A vector store is a database that stores embeddings — dense numerical representations of text. "
            "FAISS (Facebook AI Similarity Search) is a popular open-source library for efficient "
            "similarity search over millions of vectors. It enables fast nearest-neighbor retrieval "
            "used in RAG pipelines."
        ),
        metadata={"source": "vector_stores", "topic": "vector store"}
    ),
    Document(
        page_content=(
            "Large Language Models (LLMs) are neural networks trained on vast text corpora using "
            "self-supervised learning. They learn to predict the next token and develop emergent "
            "capabilities such as reasoning, code generation, and instruction following. "
            "Examples include GPT-4, Claude, LLaMA, and Mistral."
        ),
        metadata={"source": "llm_basics", "topic": "LLM"}
    ),
    Document(
        page_content=(
            "MCP (Model Context Protocol) is an open protocol developed by Anthropic that standardizes "
            "how AI models connect to external tools and data sources. It uses a host/client/server "
            "architecture and supports STDIO and HTTP transports. Servers expose tools (actions) "
            "and resources (read-only context)."
        ),
        metadata={"source": "mcp_protocol", "topic": "MCP"}
    ),
    Document(
        page_content=(
            "Machine learning is a subset of artificial intelligence that enables systems to learn "
            "from data without being explicitly programmed. The three main paradigms are supervised "
            "learning (labeled data), unsupervised learning (patterns in unlabeled data), and "
            "reinforcement learning (reward-based optimization)."
        ),
        metadata={"source": "ml_overview", "topic": "machine learning"}
    ),
    Document(
        page_content=(
            "Embeddings are dense vector representations of text that capture semantic meaning. "
            "Similar texts have similar embedding vectors (small cosine distance). "
            "Embeddings are the foundation of semantic search and RAG: a query is embedded "
            "and the nearest document embeddings are retrieved as context."
        ),
        metadata={"source": "embeddings_explained", "topic": "embeddings"}
    ),
    Document(
        page_content=(
            "An AI agent is a system that perceives its environment, plans actions, and executes "
            "tools to achieve a goal. Unlike a simple chatbot, an agent can iterate: observe → "
            "plan → act → observe. LangChain, LangGraph, and AutoGen are frameworks for building agents. "
            "Agents often use RAG for grounded knowledge retrieval."
        ),
        metadata={"source": "agents_intro", "topic": "agent"}
    ),
]

# Build FAISS vector store with fake embeddings (dimension=768)
embeddings = FakeEmbeddings(size=768)
vectorstore = FAISS.from_documents(KB_DOCS, embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

print(f"KB built ✅ — {len(KB_DOCS)} documents indexed in FAISS")
print("Topics:", [d.metadata["topic"] for d in KB_DOCS])

In [ ]:
# Quick retrieval test
test_docs = retriever.invoke("What is RAG?")
print("Test retrieval for 'What is RAG?' — top 3 docs:")
for doc in test_docs:
    print(f"  [kb:{doc.metadata['source']}] {doc.page_content[:80]}...")

---
## Exercise 2 — Add a Free External Tool (Wikipedia)

No API key required — Wikipedia is public.

In [ ]:
from langchain_community.utilities import WikipediaAPIWrapper

_wiki_api = WikipediaAPIWrapper(top_k_results=2, doc_content_chars_max=600)


def wiki_search(query: str) -> list[dict]:
    """
    Query Wikipedia and return a list of snippets with titles.
    Returns an empty list if Wikipedia is unavailable or no results found.
    """
    try:
        raw = _wiki_api.run(query)
        if not raw or "No good Wikipedia Search Result" in raw:
            return []
        # Split into individual article snippets
        snippets = []
        for chunk in raw.split("\n\n"):
            if chunk.strip():
                # First line is usually the title
                lines = chunk.strip().splitlines()
                title = lines[0].replace("Page: ", "").strip()
                content = " ".join(lines[1:])[:400].strip()
                if content:
                    snippets.append({"title": title, "content": content})
        return snippets[:2]  # max 2 snippets
    except Exception as e:
        return [{"title": "error", "content": f"Wikipedia unavailable: {e}"}]


# Test the Wikipedia tool
print("=== Wikipedia tool test: 'Eiffel Tower' ===")
results = wiki_search("Eiffel Tower")
for r in results:
    print(f"  [{r['title']}]: {r['content'][:120]}...")

---
## Exercise 3 — Rule-Based Planner

Prefer the KB for known topics; fall back to Wikipedia for everything else.

In [ ]:
# All topics covered in the knowledge base
KB_TOPICS = {
    "rag", "retrieval augmented", "retrieval-augmented",
    "langchain", "lcel",
    "vector store", "faiss", "embedding",
    "llm", "large language model",
    "mcp", "model context protocol",
    "machine learning", "supervised", "unsupervised", "reinforcement",
    "agent", "agentic",
    "neural network",
}


def planner(question: str) -> dict:
    """
    Rule-based planner.
    Returns: {use_kb: bool, use_wikipedia: bool, reason: str}
    """
    q_lower = question.lower()

    # Check if question mentions any KB topic
    matched_topics = [topic for topic in KB_TOPICS if topic in q_lower]

    if matched_topics:
        return {
            "use_kb": True,
            "use_wikipedia": False,
            "matched_topics": matched_topics,
            "reason": f"KB covers these topics: {matched_topics}"
        }
    else:
        return {
            "use_kb": False,
            "use_wikipedia": True,
            "matched_topics": [],
            "reason": "No KB match — falling back to Wikipedia"
        }


# Test the planner
test_questions = [
    "What is RAG in AI?",
    "Who built the Eiffel Tower?",
    "How does machine learning work?"
]

print("=== Planner tests ===")
for q in test_questions:
    plan = planner(q)
    route = "KB" if plan["use_kb"] else "Wikipedia"
    print(f"  Q: {q!r}")
    print(f"     → {route} | {plan['reason']}\n")

---
## Exercise 4 — Answer Function with Source Citation

Retrieve context per plan, combine sources, and generate a cited answer.
Uses `FakeListLLM` by default; optionally loads `sshleifer/tiny-gpt2`.

In [ ]:
from langchain_community.llms.fake import FakeListLLM

# Stub LLM — cycles through pre-defined responses
_stub_responses = [
    "Based on the retrieved context, here is a concise answer grounded in the provided sources.",
    "The evidence from the retrieved documents suggests the following explanation.",
    "According to the retrieved information, the answer to your question is as follows.",
    "From the available context, here is what the sources indicate about this topic.",
    "The retrieved evidence points to the following conclusion based on the knowledge base.",
    "Drawing from the retrieved sources, the answer is supported by the following context.",
]
stub_llm = FakeListLLM(responses=_stub_responses)


def load_real_llm():
    """Load tiny-gpt2 for local generation (CPU-friendly)."""
    from transformers import pipeline
    from langchain_community.llms import HuggingFacePipeline
    pipe = pipeline("text-generation", model="sshleifer/tiny-gpt2",
                    max_new_tokens=80, do_sample=False)
    return HuggingFacePipeline(pipeline=pipe)


llm = load_real_llm() if USE_REAL_LLM else stub_llm
print(f"LLM: {'tiny-gpt2 (real)' if USE_REAL_LLM else 'FakeListLLM (stub)'} ✅")

In [ ]:
def answer(question: str) -> dict:
    """
    Full RAG answer pipeline:
    1. Plan (KB vs Wikipedia)
    2. Retrieve context
    3. Generate answer with stub/real LLM
    4. Cite all sources as [kb:source] or [wiki:title]
    5. Handle thin evidence gracefully
    """
    plan = planner(question)
    sources = []
    context_parts = []

    # --- Retrieve from KB ---
    if plan["use_kb"]:
        kb_docs = retriever.invoke(question)
        for doc in kb_docs:
            src = f"kb:{doc.metadata['source']}"
            sources.append(src)
            context_parts.append(f"[{src}] {doc.page_content}")

    # --- Retrieve from Wikipedia ---
    if plan["use_wikipedia"]:
        wiki_snippets = wiki_search(question)
        for snippet in wiki_snippets:
            # Sanitize title for citation key
            title_key = snippet["title"].replace(" ", "_")[:40]
            src = f"wiki:{title_key}"
            sources.append(src)
            context_parts.append(f"[{src}] {snippet['content']}")

    # --- Handle missing evidence ---
    if not context_parts:
        return {
            "question": question,
            "plan": plan,
            "sources": [],
            "answer": (
                "I don't have enough evidence to answer this question reliably. "
                "Try rephrasing or asking something more specific. "
                "Suggested follow-up: could you provide more context or keywords?"
            )
        }

    # --- Build prompt ---
    context_str = "\n\n".join(context_parts)
    prompt = (
        f"Answer the question based ONLY on the following context. "
        f"Cite sources inline using the bracket notation.\n\n"
        f"Context:\n{context_str}\n\n"
        f"Question: {question}\n\n"
        f"Answer (cite sources):"
    )

    # --- Generate answer ---
    if USE_REAL_LLM:
        generated = llm.invoke(prompt)
    else:
        # Stub: construct a meaningful answer from context
        generated = stub_llm.invoke(prompt)
        # Augment stub answer with first context snippet + citations
        first_context = context_parts[0].split("]", 1)[-1].strip()[:250]
        cite_str = " ".join(f"[{s}]" for s in sources[:3])
        generated = f"{first_context}... {cite_str}"

    return {
        "question": question,
        "plan": plan,
        "sources": sources,
        "context_preview": context_parts[0][:150] + "...",
        "answer": generated
    }


print("Answer function defined ✅")

---
## Exercise 5 — Quick Check: 3 Sample Questions

One KB-covered, one external (Wikipedia), one ambiguous.

In [ ]:
QUESTIONS = [
    "What is RAG and how does it work in AI systems?",      # KB-covered
    "When was the Eiffel Tower built and by whom?",         # External (Wikipedia)
    "How does machine learning relate to neural networks?", # Ambiguous (both KB topics)
]

separator = "=" * 60

for q in QUESTIONS:
    result = answer(q)
    print(separator)
    print(f"QUESTION : {result['question']}")
    print(f"PLAN     : {result['plan']['reason']}")
    print(f"SOURCES  : {result['sources']}")
    if 'context_preview' in result:
        print(f"CONTEXT  : {result['context_preview']}")
    print(f"ANSWER   : {result['answer']}")
    print()

---
## Bonus — Test Missing Evidence Handling

In [ ]:
# Edge case: very obscure question with no KB match and poor Wikipedia result
edge_result = answer("What is the recipe for my grandmother's apple pie?")
print(f"QUESTION : {edge_result['question']}")
print(f"PLAN     : {edge_result['plan']['reason']}")
print(f"SOURCES  : {edge_result['sources']}")
print(f"ANSWER   : {edge_result['answer']}")

---
## Summary

| Exercise | Built | Status |
|----------|-------|--------|
| 1 | 8-doc FAISS KB with FakeEmbeddings, top-3 retriever | ✅ |
| 2 | Wikipedia tool — `wiki_search(query)` returning `[{title, content}]` | ✅ |
| 3 | Rule-based planner — KB topics matched via keyword set | ✅ |
| 4 | `answer()` — retrieve → cite `[kb:x]`/`[wiki:x]` → FakeListLLM stub | ✅ |
| 5 | 3 questions: KB-covered / external / ambiguous + missing evidence test | ✅ |

### Agent Flow Diagram

```
Question
  └──► planner(q)
            │
            ├── use_kb=True ──► retriever.invoke(q)  → [kb:source] docs
            │
            └── use_wikipedia=True ──► wiki_search(q) → [wiki:title] snippets
                                                │
                                    context = KB docs + wiki snippets
                                                │
                                          llm.invoke(prompt)
                                                │
                                    Answer with inline citations
```

### Key Design Decisions

- **FakeEmbeddings**: makes FAISS work without an embedding API key — ideal for prototyping
- **Rule-based planner**: deterministic and debuggable — no LLM needed for routing
- **Source citation format**: `[kb:source]` and `[wiki:title]` — distinguishes KB from external
- **Missing evidence fallback**: explicit message + suggested follow-up instead of hallucinating